# DAMICORE artifacts: violence against women

This notebook is the only database-facing preparation stage for this hypothesis.
It creates the three corpora and the metadata contract consumed by the experiment
notebooks. It does not execute DAMICORE or interpret clusters.

**Scope:** distinct reports with a female victim registered from January 2020 through
June 2026, restricted to the exploratory violence-related source taxonomy.


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.create_artifacts import (
    configured_database_url,
    initialize_preparation,
    prepare_case_artifacts,
    prepare_normalized_artifacts,
    prepare_source_data,
    write_artifact_manifests,
)
from hypotheses.violence_against_women.scripts.experiment_common import CASE_SEEDS

load_dotenv(PROJECT_ROOT / ".env")
CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
ARTIFACT_WORKERS = 2
DATABASE_URL = configured_database_url()
PATHS = initialize_preparation(CATEGORY_SET_VERSION, ARTIFACT_WORKERS)


/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Query scope and shared context counts

`source_hash` is used only inside PostgreSQL for distinct-report counts. It is not
written to the derived artifacts.


In [ ]:
coverage, context_counts, prepared_case_records = prepare_source_data(DATABASE_URL, PATHS)
display(coverage)
display(context_counts.head())
print(f"Context rows: {len(context_counts):,}")
del coverage


## 2. Select eligible categories and create the normalized corpus

The selection rule follows the source taxonomy and the minimum support threshold used
by the original experiment. Category order is fixed here and reused by every notebook.


In [ ]:
category_order, included_support, category_map, normalized_corpus_bytes = prepare_normalized_artifacts(
    context_counts, PATHS
)
del context_counts
display(category_map)
print(f"Included categories: {len(category_order)}")
print(f"Normalized corpus bytes per category: {normalized_corpus_bytes:,}")


## 3. Create the case-full and case-balanced corpora

The case grain is `source_hash + category` in memory. Canonical case documents contain
only the 20 contextual dimensions, so report identifiers are not exported.


In [ ]:
case_category_map, balanced_sample_size, case_count = prepare_case_artifacts(
    prepared_case_records,
    category_order,
    included_support,
    category_map,
    PATHS,
    workers=ARTIFACT_WORKERS,
)
del prepared_case_records, included_support, category_map
print(f"Case records in memory: {case_count:,}")
print(f"Balanced sample size per category and replica: {balanced_sample_size:,}")
print(f"Balanced replicas: {len(CASE_SEEDS)}")
display(case_category_map.head())


## 4. Write the artifact manifest

The manifest is the compatibility contract for the three experiment notebooks.


In [ ]:
manifest = write_artifact_manifests(PATHS, category_order, balanced_sample_size)
print("Artifact manifest written.")
